In [33]:
import pandas as pd

# Veriyi yükleme
file_path = './merged_df_2.xlsx'
data = pd.read_excel(file_path)

# Verinin genel yapısını inceleme
print(data.head())
print(data.info())
print(data.describe())


   id  age  gender  nobet_tipi  nobet_frequnency  disease_duration  \
0   0   52       2           3                 1               3.0   
1   1   19       2           3                 3               8.0   
2   2   55       2           3                 1               0.4   
3   3   29       2           2                 3              25.0   
4   4   71       2           2                 3               6.0   

   marital_status  se_history  nbt_uyku  nbt_uyaniklik  head_trauma  \
0               1           0         1              0            0   
1               0           0         0              1            1   
2               1           1         0              1            0   
3               0           0         0              1            1   
4               1           0         0              1            1   

   interceptive_delivery  febril_convulsion  epilepsy_family_history  \
0                      0                  0                        0   
1       

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

# TF-IDF vektörizasyonu
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(data['eeg_sonucu'])

# PCA ile boyut indirgeme
pca = PCA(n_components=1)
eeg_numeric = pca.fit_transform(tfidf_matrix.toarray())

# Yeni nümerik değeri veri setine ekleme
data['eeg_numeric'] = eeg_numeric
data = data.drop(columns=['eeg_sonucu'])


In [35]:
from imblearn.over_sampling import SMOTE

# Input ve output ayrıştırma
X = data.drop(columns=['sinif'])
y = data['sinif']

# SMOTE kullanarak veri artırma
smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)


In [36]:
nan_rows = X[X.isnull().any(axis=1)]
print(nan_rows)



Empty DataFrame
Columns: [id, age, gender, nobet_tipi, nobet_frequnency, disease_duration, marital_status, se_history, nbt_uyku, nbt_uyaniklik, head_trauma, interceptive_delivery, febril_convulsion, epilepsy_family_history, fk_fever_nbt_family, family_history, comorbid_diseases, eeg_numeric]
Index: []


In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import f1_score, recall_score, accuracy_score

# Veri setini PyTorch tensörlerine dönüştürme
X_tensor = torch.tensor(X_resampled.values, dtype=torch.float32)
y_tensor = torch.tensor(y_resampled.values, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# MLP modeli oluşturma
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

input_dim = X_tensor.shape[1]
hidden_dim = 128
output_dim = len(data['sinif'].unique())

model = MLP(input_dim, hidden_dim, output_dim)

# Loss ve optimizer belirleme
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Modeli eğitme
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

# Modeli değerlendirme fonksiyonu
def evaluate_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    f1 = f1_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    accuracy = accuracy_score(all_labels, all_preds)
    return f1, recall, accuracy

f1, recall, accuracy = evaluate_model(model, test_dataloader)
print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


/home/cevher/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/10, Loss: 2.17598557472229
Epoch 2/10, Loss: 2.2344841957092285
Epoch 3/10, Loss: 2.7728638648986816
Epoch 4/10, Loss: 2.445946216583252
Epoch 5/10, Loss: 2.4115426540374756
Epoch 6/10, Loss: 2.2190756797790527
Epoch 7/10, Loss: 1.9628913402557373
Epoch 8/10, Loss: 2.0546483993530273
Epoch 9/10, Loss: 2.303062677383423
Epoch 10/10, Loss: 2.2093095779418945
F1 Score: 0.14101779571708598, Recall: 0.19767441860465115, Accuracy: 0.19767441860465115


In [38]:
from sklearn.preprocessing import StandardScaler

# Özelliklerin normalizasyonu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)

# Veri setini PyTorch tensörlerine dönüştürme
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_resampled.values, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Daha derin bir MLP modeli oluşturma
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DeepMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc4(x)
        return x

input_dim = X_tensor.shape[1]
hidden_dim = 256
output_dim = len(data['sinif'].unique())

model = DeepMLP(input_dim, hidden_dim, output_dim)

# Loss ve optimizer belirleme
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Modeli eğitme
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

# Modeli değerlendirme fonksiyonu
f1, recall, accuracy = evaluate_model(model, test_dataloader)
print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


Epoch 1/20, Loss: 1.9637446403503418
Epoch 2/20, Loss: 1.5835307836532593
Epoch 3/20, Loss: 1.6452271938323975
Epoch 4/20, Loss: 1.4724019765853882
Epoch 5/20, Loss: 1.0091787576675415
Epoch 6/20, Loss: 1.3842328786849976
Epoch 7/20, Loss: 1.318989634513855
Epoch 8/20, Loss: 1.5171864032745361
Epoch 9/20, Loss: 1.2736223936080933
Epoch 10/20, Loss: 1.2624406814575195
Epoch 11/20, Loss: 1.3042902946472168
Epoch 12/20, Loss: 1.3788219690322876
Epoch 13/20, Loss: 1.4372844696044922
Epoch 14/20, Loss: 1.2006046772003174
Epoch 15/20, Loss: 1.2041815519332886
Epoch 16/20, Loss: 1.2617794275283813
Epoch 17/20, Loss: 1.0519602298736572
Epoch 18/20, Loss: 0.9096380472183228
Epoch 19/20, Loss: 0.9899694323539734
Epoch 20/20, Loss: 1.2035280466079712
F1 Score: 0.5646013016014485, Recall: 0.5930232558139535, Accuracy: 0.5930232558139535


In [39]:
import optuna
from sklearn.metrics import f1_score, recall_score, accuracy_score

# MLP modeli
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(DeepMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Optuna ile hiperparametre optimizasyonu
def objective(trial):
    input_dim = X_tensor.shape[1]
    hidden_dim = trial.suggest_int('hidden_dim', 64, 512)
    hidden_layers = trial.suggest_int('hidden_layers', 1, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2)

    model = DeepMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    num_epochs = 10
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    f1, recall, accuracy = evaluate_model(model, test_dataloader)
    return f1

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print('Best trial:')
trial = study.best_trial
print(f'F1 Score: {trial.value}')
print('Best hyperparameters: ', trial.params)

# En iyi hiperparametrelerle model eğitme
best_params = trial.params
model = DeepMLP(input_dim, best_params['hidden_dim'], best_params['hidden_layers'], output_dim, best_params['dropout_rate'])

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best_params['lr'])

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

f1, recall, accuracy = evaluate_model(model, test_dataloader)
print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


[I 2024-12-02 09:53:53,161] A new study created in memory with name: no-name-c59b2788-4fbf-44a6-b7b9-e94ac74347c6
[I 2024-12-02 09:53:55,355] Trial 0 finished with value: 0.4422925340035587 and parameters: {'hidden_dim': 191, 'hidden_layers': 4, 'dropout_rate': 0.33450494883137083, 'lr': 0.008066070939374026}. Best is trial 0 with value: 0.4422925340035587.
[I 2024-12-02 09:53:57,948] Trial 1 finished with value: 0.2631951503981763 and parameters: {'hidden_dim': 84, 'hidden_layers': 5, 'dropout_rate': 0.38879276487394454, 'lr': 0.008408694169887541}. Best is trial 0 with value: 0.4422925340035587.
[I 2024-12-02 09:54:00,586] Trial 2 finished with value: 0.4414243933529818 and parameters: {'hidden_dim': 298, 'hidden_layers': 3, 'dropout_rate': 0.46729019331556443, 'lr': 0.007924230943264401}. Best is trial 0 with value: 0.4422925340035587.
[I 2024-12-02 09:54:03,203] Trial 3 finished with value: 0.5347156961197971 and parameters: {'hidden_dim': 169, 'hidden_layers': 4, 'dropout_rate': 0

Best trial:
F1 Score: 0.6735017622126661
Best hyperparameters:  {'hidden_dim': 481, 'hidden_layers': 3, 'dropout_rate': 0.13600915999019414, 'lr': 0.0023049646387200504}
Epoch 1/20, Loss: 1.629575252532959
Epoch 2/20, Loss: 1.2765326499938965
Epoch 3/20, Loss: 1.1202495098114014
Epoch 4/20, Loss: 0.7453368306159973
Epoch 5/20, Loss: 0.7780599594116211
Epoch 6/20, Loss: 0.49197259545326233
Epoch 7/20, Loss: 1.0229008197784424
Epoch 8/20, Loss: 0.5807236433029175
Epoch 9/20, Loss: 0.12429641932249069
Epoch 10/20, Loss: 0.7960814833641052
Epoch 11/20, Loss: 0.344135046005249
Epoch 12/20, Loss: 0.5446134209632874
Epoch 13/20, Loss: 0.6489229798316956
Epoch 14/20, Loss: 0.3694339394569397
Epoch 15/20, Loss: 0.24165673553943634
Epoch 16/20, Loss: 0.4012458920478821
Epoch 17/20, Loss: 0.4455524981021881
Epoch 18/20, Loss: 0.23145070672035217
Epoch 19/20, Loss: 0.2434263825416565
Epoch 20/20, Loss: 0.24635577201843262
F1 Score: 0.6922784475534755, Recall: 0.6976744186046512, Accuracy: 0.697674

In [40]:
import numpy as np

# Geliştirilmiş MLP modeli
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(ImprovedMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_dim = X_tensor.shape[1]
hidden_dim = best_params['hidden_dim']
hidden_layers = best_params['hidden_layers']
dropout_rate = best_params['dropout_rate']
lr = best_params['lr']

model = ImprovedMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Erken durdurma için parametreler
patience = 5
best_loss = np.inf
epochs_no_improve = 0

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    average_loss = running_loss / len(train_dataloader)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {average_loss}')

    # Validation loss'u kontrol etme
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_dataloader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    val_loss /= len(test_dataloader)

    if val_loss < best_loss:
        best_loss = val_loss
        epochs_no_improve = 0
        best_model = model.state_dict()
    else:
        epochs_no_improve += 1

    if epochs_no_improve == patience:
        print('Early stopping!')
        break

# En iyi modeli yükleme
model.load_state_dict(best_model)

f1, recall, accuracy = evaluate_model(model, test_dataloader)
print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


Epoch 1/50, Loss: 1.5975268940592922
Epoch 2/50, Loss: 1.200409495553305
Epoch 3/50, Loss: 1.0644348333048266
Epoch 4/50, Loss: 0.9695903617282247
Epoch 5/50, Loss: 0.8675129434397054
Epoch 6/50, Loss: 0.7934855118740437
Epoch 7/50, Loss: 0.7494529097579247
Epoch 8/50, Loss: 0.6526216490324154
Epoch 9/50, Loss: 0.6757142176461775
Epoch 10/50, Loss: 0.6531550925831462
Epoch 11/50, Loss: 0.5873301119305366
Epoch 12/50, Loss: 0.5188828488421995
Early stopping!
F1 Score: 0.6149578182433942, Recall: 0.625, Accuracy: 0.625


## Ensemble bagging 

In [41]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import f1_score, recall_score, accuracy_score

# Improved MLP modeli
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(ImprovedMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_dim = X_tensor.shape[1]
hidden_dim = best_params['hidden_dim']
hidden_layers = best_params['hidden_layers']
dropout_rate = best_params['dropout_rate']
lr = best_params['lr']
output_dim = len(data['sinif'].unique())

# Bagging için birden fazla model eğitme
num_models = 5
models = []

for _ in range(num_models):
    model = ImprovedMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    num_epochs = 20
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    models.append(model)

# Bagging tahminleri birleştirme
def bagging_predict(models, dataloader):
    all_preds = []
    with torch.no_grad():
        for inputs, _ in dataloader:
            preds = [model(inputs) for model in models]
            avg_preds = torch.mean(torch.stack(preds), dim=0)
            _, final_preds = torch.max(avg_preds, 1)
            all_preds.extend(final_preds.numpy())
    return all_preds

# Test seti üzerinde bagging tahminleri
all_labels = []
for _, labels in test_dataloader:
    all_labels.extend(labels.numpy())

bagging_preds = bagging_predict(models, test_dataloader)

# Performans ölçütleri
f1 = f1_score(all_labels, bagging_preds, average='weighted')
recall = recall_score(all_labels, bagging_preds, average='weighted')
accuracy = accuracy_score(all_labels, bagging_preds)

print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


F1 Score: 0.6451803183293195, Recall: 0.6540697674418605, Accuracy: 0.6540697674418605


In [42]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Geliştirilmiş MLP modeli (baz model)
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(ImprovedMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Temel modelleri eğitme
input_dim = X_tensor.shape[1]
hidden_dim = best_params['hidden_dim']
hidden_layers = best_params['hidden_layers']
dropout_rate = best_params['dropout_rate']
lr = best_params['lr']
output_dim = len(data['sinif'].unique())

def train_mlp_model(X_train, y_train, X_val, y_val):
    model = ImprovedMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    num_epochs = 20
    best_model = None
    best_val_loss = np.inf

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True):
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(X_val)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

    model.load_state_dict(best_model)
    return model

# Veriyi eğitim ve doğrulama setlerine ayırma
X_train_val, X_test, y_train_val, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)

# Baz modelleri eğitme
mlp_model = train_mlp_model(X_train, y_train, X_val, y_val)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_val.numpy(), y_train_val.numpy())
svm_model = SVC(probability=True, random_state=42).fit(X_train_val.numpy(), y_train_val.numpy())

# Tahminleri meta model için hazırlama
def get_predictions(models, X):
    mlp_model, rf_model, svm_model = models
    mlp_model.eval()
    with torch.no_grad():
        mlp_preds = mlp_model(X).numpy()
    rf_preds = rf_model.predict_proba(X.numpy())
    svm_preds = svm_model.predict_proba(X.numpy())
    return np.hstack([mlp_preds, rf_preds, svm_preds])

meta_train_preds = get_predictions([mlp_model, rf_model, svm_model], X_train_val)
meta_test_preds = get_predictions([mlp_model, rf_model, svm_model], X_test)

# Meta model eğitme
meta_model = LogisticRegression(random_state=42)
meta_model.fit(meta_train_preds, y_train_val.numpy())

# Meta model ile tahmin yapma
meta_preds = meta_model.predict(meta_test_preds)

# Performans ölçütleri
f1 = f1_score(y_test.numpy(), meta_preds, average='weighted')
recall = recall_score(y_test.numpy(), meta_preds, average='weighted')
accuracy = accuracy_score(y_test.numpy(), meta_preds)

print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


F1 Score: 0.784554492449454, Recall: 0.7877906976744186, Accuracy: 0.7877906976744186


/home/cevher/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Higher epochs with earlier methods

In [43]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Geliştirilmiş MLP modeli (baz model)
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(ImprovedMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Temel modelleri eğitme
input_dim = X_tensor.shape[1]
hidden_dim = best_params['hidden_dim']
hidden_layers = best_params['hidden_layers']
dropout_rate = best_params['dropout_rate']
lr = best_params['lr']
output_dim = len(data['sinif'].unique())

def train_mlp_model(X_train, y_train, X_val, y_val):
    model = ImprovedMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    num_epochs = 200
    best_model = None
    best_val_loss = np.inf

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True):
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(X_val)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

    model.load_state_dict(best_model)
    return model

# Veriyi eğitim ve doğrulama setlerine ayırma
X_train_val, X_test, y_train_val, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)

# Baz modelleri eğitme
mlp_model = train_mlp_model(X_train, y_train, X_val, y_val)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_val.numpy(), y_train_val.numpy())
svm_model = SVC(probability=True, random_state=42).fit(X_train_val.numpy(), y_train_val.numpy())

# Tahminleri meta model için hazırlama
def get_predictions(models, X):
    mlp_model, rf_model, svm_model = models
    mlp_model.eval()
    with torch.no_grad():
        mlp_preds = mlp_model(X).numpy()
    rf_preds = rf_model.predict_proba(X.numpy())
    svm_preds = svm_model.predict_proba(X.numpy())
    return np.hstack([mlp_preds, rf_preds, svm_preds])

meta_train_preds = get_predictions([mlp_model, rf_model, svm_model], X_train_val)
meta_test_preds = get_predictions([mlp_model, rf_model, svm_model], X_test)

# Meta model eğitme
meta_model = LogisticRegression(random_state=42)
meta_model.fit(meta_train_preds, y_train_val.numpy())

# Meta model ile tahmin yapma
meta_preds = meta_model.predict(meta_test_preds)

# Performans ölçütleri
f1 = f1_score(y_test.numpy(), meta_preds, average='weighted')
recall = recall_score(y_test.numpy(), meta_preds, average='weighted')
accuracy = accuracy_score(y_test.numpy(), meta_preds)

print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


F1 Score: 0.7849795118722939, Recall: 0.7877906976744186, Accuracy: 0.7877906976744186


/home/cevher/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import f1_score, recall_score, accuracy_score

# Improved MLP modeli
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(ImprovedMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_dim = X_tensor.shape[1]
hidden_dim = best_params['hidden_dim']
hidden_layers = best_params['hidden_layers']
dropout_rate = best_params['dropout_rate']
lr = best_params['lr']
output_dim = len(data['sinif'].unique())

# Bagging için birden fazla model eğitme
num_models = 5
models = []

for _ in range(num_models):
    model = ImprovedMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    num_epochs = 200
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    models.append(model)

# Bagging tahminleri birleştirme
def bagging_predict(models, dataloader):
    all_preds = []
    with torch.no_grad():
        for inputs, _ in dataloader:
            preds = [model(inputs) for model in models]
            avg_preds = torch.mean(torch.stack(preds), dim=0)
            _, final_preds = torch.max(avg_preds, 1)
            all_preds.extend(final_preds.numpy())
    return all_preds

# Test seti üzerinde bagging tahminleri
all_labels = []
for _, labels in test_dataloader:
    all_labels.extend(labels.numpy())

bagging_preds = bagging_predict(models, test_dataloader)

# Performans ölçütleri
f1 = f1_score(all_labels, bagging_preds, average='weighted')
recall = recall_score(all_labels, bagging_preds, average='weighted')
accuracy = accuracy_score(all_labels, bagging_preds)

print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


F1 Score: 0.7199491746974341, Recall: 0.7238372093023255, Accuracy: 0.7238372093023255


In [45]:
import optuna
from sklearn.metrics import f1_score, recall_score, accuracy_score

# MLP modeli
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(DeepMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Optuna ile hiperparametre optimizasyonu
def objective(trial):
    input_dim = X_tensor.shape[1]
    hidden_dim = trial.suggest_int('hidden_dim', 64, 512)
    hidden_layers = trial.suggest_int('hidden_layers', 1, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2)

    model = DeepMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    num_epochs = 10
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    f1, recall, accuracy = evaluate_model(model, test_dataloader)
    return f1

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print('Best trial:')
trial = study.best_trial
print(f'F1 Score: {trial.value}')
print('Best hyperparameters: ', trial.params)

# En iyi hiperparametrelerle model eğitme
best_params = trial.params
model = DeepMLP(input_dim, best_params['hidden_dim'], best_params['hidden_layers'], output_dim, best_params['dropout_rate'])

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best_params['lr'])

num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

f1, recall, accuracy = evaluate_model(model, test_dataloader)
print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


[I 2024-12-02 10:04:48,912] A new study created in memory with name: no-name-0c57af3e-368c-4549-9b3a-1cf093aca8c0
[I 2024-12-02 10:04:49,669] Trial 0 finished with value: 0.5328862236233061 and parameters: {'hidden_dim': 198, 'hidden_layers': 1, 'dropout_rate': 0.4136296019297976, 'lr': 0.0020434864004539536}. Best is trial 0 with value: 0.5328862236233061.
[I 2024-12-02 10:04:50,804] Trial 1 finished with value: 0.634543466896757 and parameters: {'hidden_dim': 182, 'hidden_layers': 2, 'dropout_rate': 0.20813587160498095, 'lr': 0.007846738930206633}. Best is trial 1 with value: 0.634543466896757.
[I 2024-12-02 10:04:54,547] Trial 2 finished with value: 0.5407955921843061 and parameters: {'hidden_dim': 106, 'hidden_layers': 4, 'dropout_rate': 0.11944689109058992, 'lr': 0.002289251592261959}. Best is trial 1 with value: 0.634543466896757.
[I 2024-12-02 10:04:56,060] Trial 3 finished with value: 0.5642401618025786 and parameters: {'hidden_dim': 105, 'hidden_layers': 1, 'dropout_rate': 0.1

Best trial:
F1 Score: 0.6861637021403112
Best hyperparameters:  {'hidden_dim': 479, 'hidden_layers': 2, 'dropout_rate': 0.14819361651463414, 'lr': 0.003329839660821326}
Epoch 1/200, Loss: 1.4253138303756714
Epoch 2/200, Loss: 1.26954185962677
Epoch 3/200, Loss: 1.1398687362670898
Epoch 4/200, Loss: 0.767065167427063
Epoch 5/200, Loss: 0.9841107130050659
Epoch 6/200, Loss: 0.6737945675849915
Epoch 7/200, Loss: 0.6285619139671326
Epoch 8/200, Loss: 0.39574629068374634
Epoch 9/200, Loss: 0.4770589768886566
Epoch 10/200, Loss: 0.3358553647994995
Epoch 11/200, Loss: 0.6408332586288452
Epoch 12/200, Loss: 0.4081467092037201
Epoch 13/200, Loss: 0.4886099696159363
Epoch 14/200, Loss: 0.3849365711212158
Epoch 15/200, Loss: 0.3595990538597107
Epoch 16/200, Loss: 0.2327037900686264
Epoch 17/200, Loss: 0.13522663712501526
Epoch 18/200, Loss: 0.28100305795669556
Epoch 19/200, Loss: 0.14457719027996063
Epoch 20/200, Loss: 0.20321737229824066
Epoch 21/200, Loss: 0.4100428521633148
Epoch 22/200, Loss:

## XGBoost

In [46]:
import xgboost as xgb
from sklearn.neighbors import KNeighborsClassifier

# Geliştirilmiş MLP modeli (baz model)
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate):
        super(ImprovedMLP, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Temel modelleri eğitme
input_dim = X_tensor.shape[1]
hidden_dim = best_params['hidden_dim']
hidden_layers = best_params['hidden_layers']
dropout_rate = best_params['dropout_rate']
lr = best_params['lr']
output_dim = len(data['sinif'].unique())

def train_mlp_model(X_train, y_train, X_val, y_val):
    model = ImprovedMLP(input_dim, hidden_dim, hidden_layers, output_dim, dropout_rate)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    num_epochs = 20
    best_model = None
    best_val_loss = np.inf

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True):
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(X_val)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

    model.load_state_dict(best_model)
    return model

# Veriyi eğitim ve doğrulama setlerine ayırma
X_train_val, X_test, y_train_val, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)

# Baz modelleri eğitme
mlp_model = train_mlp_model(X_train, y_train, X_val, y_val)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_val.numpy(), y_train_val.numpy())
svm_model = SVC(probability=True, random_state=42).fit(X_train_val.numpy(), y_train_val.numpy())
knn_model = KNeighborsClassifier(n_neighbors=5).fit(X_train_val.numpy(), y_train_val.numpy())

# Tahminleri meta model için hazırlama
def get_predictions(models, X):
    mlp_model, rf_model, svm_model, knn_model = models
    mlp_model.eval()
    with torch.no_grad():
        mlp_preds = mlp_model(X).numpy()
    rf_preds = rf_model.predict_proba(X.numpy())
    svm_preds = svm_model.predict_proba(X.numpy())
    knn_preds = knn_model.predict_proba(X.numpy())
    return np.hstack([mlp_preds, rf_preds, svm_preds, knn_preds])

meta_train_preds = get_predictions([mlp_model, rf_model, svm_model, knn_model], X_train_val)
meta_test_preds = get_predictions([mlp_model, rf_model, svm_model, knn_model], X_test)

# Meta model eğitme
meta_model = xgb.XGBClassifier(random_state=42)
meta_model.fit(meta_train_preds, y_train_val.numpy())

# Meta model ile tahmin yapma
meta_preds = meta_model.predict(meta_test_preds)

# Performans ölçütleri
f1 = f1_score(y_test.numpy(), meta_preds, average='weighted')
recall = recall_score(y_test.numpy(), meta_preds, average='weighted')
accuracy = accuracy_score(y_test.numpy(), meta_preds)

print(f'F1 Score: {f1}, Recall: {recall}, Accuracy: {accuracy}')


/home/cevher/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


F1 Score: 0.5241842596908101, Recall: 0.49709302325581395, Accuracy: 0.49709302325581395
